# ATC Multi-Agent GRPO Training

**Before running:** Push your local branch to GitHub first.
```
git add -A && git commit -m 'pre-colab sync' && git push origin yashh
```

**Recommended runtime:** `Runtime → Change runtime type → GPU (T4 or better)`

**Run order:** Execute cells top-to-bottom. Eval/plot cells at the end are optional.

In [ ]:
# ── CONFIG — edit if needed, then run all cells ───────────────────────────────
REPO_URL   = "https://github.com/GTsingh600/ats.git"
BRANCH     = "yashh"
REPO_DIR   = "/content/ATC"
USE_DRIVE  = True
OUTPUT_DIR = "/content/drive/MyDrive/atc-grpo" if USE_DRIVE else "/content/atc-grpo"

# Model + training hyperparams
MODEL_NAME    = "Qwen/Qwen2.5-7B-Instruct"
EPISODES      = 50       # 50 ≈ 25-35 min on T4. Use 200 for full run.
LORA_RANK     = 16
SEED          = 42
# T4 (16GB): N_GENERATIONS=2 is safe. L4/A100: use 4.
N_GENERATIONS = 2
EVAL_EPISODES = 3        # episodes per task for before/after eval

import os
os.environ["WANDB_MODE"]                      = "disabled"
os.environ["TOKENIZERS_PARALLELISM"]          = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

print(f"REPO_URL   : {REPO_URL}")
print(f"BRANCH     : {BRANCH}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"EPISODES   : {EPISODES}")
print(f"N_GEN      : {N_GENERATIONS}")

In [ ]:
# ── GPU CHECK ─────────────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU (T4 or better).")
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU     : {gpu}")
print(f"VRAM    : {vram:.1f} GB")
print(f"CUDA    : {torch.version.cuda}")
print(f"PyTorch : {torch.__version__}")
if vram < 12:
    print("WARNING: < 12 GB VRAM. Reduce EPISODES or switch to L4/A100.")

In [ ]:
# ── DRIVE MOUNT ───────────────────────────────────────────────────────────────
from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Output ready: {OUTPUT_DIR}")

In [ ]:
# ── INSTALL DEPENDENCIES ──────────────────────────────────────────────────────
# unsloth MUST be installed first — it pins torch/transformers versions
# that are compatible with the pre-installed Colab CUDA driver.
# Never pin trl separately; let unsloth choose a compatible version.
import subprocess, sys

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    print("  $", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])  # show last 2k chars of error
        raise RuntimeError(f"pip install failed: {args[0]}")

print("[1/3] Installing unsloth (pins compatible torch/transformers)...")
pip("unsloth")

print("[2/3] Installing training stack...")
pip("trl", "peft", "accelerate", "bitsandbytes",
    "datasets>=2.20.0", "matplotlib>=3.9.0", "numpy>=1.26.0", "openai>=1.0.0")

print("[3/3] Verifying key imports...")
from unsloth import FastLanguageModel          # noqa
from trl import GRPOTrainer, GRPOConfig        # noqa
print("  unsloth  : OK")
print("  trl GRPO : OK")

In [ ]:
# ── CLONE REPO + SETUP PATHS ──────────────────────────────────────────────────
import shutil, subprocess, os, sys
from pathlib import Path

repo = Path(REPO_DIR)
if repo.exists():
    shutil.rmtree(repo)

print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
r = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
    capture_output=True, text=True
)
if r.returncode != 0:
    # depth=1 with branch can fail if branch not on remote — try without depth
    print("  shallow clone failed, trying full clone + checkout...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory : {os.getcwd()}")
print(f"Branch            : ", end="")
b = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                   capture_output=True, text=True, cwd=REPO_DIR)
print(b.stdout.strip())
print(f"Latest commit     : ", end="")
c = subprocess.run(["git", "log", "--oneline", "-1"],
                   capture_output=True, text=True, cwd=REPO_DIR)
print(c.stdout.strip())

In [ ]:
# ── SANITY CHECK — imports + dataset ─────────────────────────────────────────
import torch
from training.dataset import build_episode_dataset
from multi_agent.environment import MultiAgentATCEnvironment
from multi_agent.generator import ChallengeGenerator
from tasks import task_catalog, ordered_tasks

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Tasks in catalog: {len(task_catalog())}")

print("\nBuilding 4-episode dataset (smoke test — ~5 seconds)...")
samples = build_episode_dataset(n_episodes=4, seed=0,
                                include_generator=True, include_supervisor=True)
roles = sorted({s["agent_role"] for s in samples})
print(f"  Samples : {len(samples)}")
print(f"  Roles   : {roles}")
assert len(samples) > 0, "Dataset is empty — check task_catalog()"

print("\nAll imports OK.")

In [ ]:
# ── HEURISTIC BASELINE (no model, no GPU) ────────────────────────────────────
# Run this before training to confirm environment is working and
# to record the pre-training reference score.
from training.train_grpo import _quick_heuristic_eval, _save_json
from pathlib import Path

print("Running heuristic baseline (deterministic planner, no LLM)...")
baseline = _quick_heuristic_eval(n_episodes=6)

_save_json(baseline, Path(OUTPUT_DIR) / "baseline_metrics.json")

print(f"\n  Composite score : {baseline['mean_composite']:.3f}")
print(f"  AMAN reward     : {baseline['mean_aman_reward']:.3f}")
print(f"  DMAN reward     : {baseline['mean_dman_reward']:.3f}")
print(f"  Mean conflicts  : {baseline['mean_conflicts']:.1f}")
print(f"  Per-episode     : {baseline['scores']}")
print(f"\nSaved to {OUTPUT_DIR}/baseline_metrics.json")

In [ ]:
# ── GRPO TRAINING ─────────────────────────────────────────────────────────────
# Runs in-process so print output streams directly into this cell.
#
# T4 memory profile (16 GB VRAM):
#   Qwen2.5-7B 4-bit : ~4.5 GB
#   LoRA rank=16     : ~0.1 GB
#   GRPO batch=2     : ~4-6 GB (2 x 512-token completions x N_GENERATIONS)
#   Total            : ~10-12 GB  →  fits with headroom
#
# If OOM: reduce EPISODES (shorter dataset) or set N_GENERATIONS=1.

import training.train_grpo as _tg
from training.train_grpo import train as grpo_train

# Patch module-level constants before calling train()
# N_GENERATIONS must divide BATCH_SIZE evenly
_tg.N_GENERATIONS = N_GENERATIONS
_tg.BATCH_SIZE    = max(2, N_GENERATIONS)  # ensures batch % n_gen == 0

print(f"Starting GRPO training")
print(f"  Model      : {MODEL_NAME}")
print(f"  Episodes   : {EPISODES}")
print(f"  LoRA rank  : {LORA_RANK}")
print(f"  Generations: {N_GENERATIONS}")
print(f"  Batch size : {_tg.BATCH_SIZE}  (effective = {_tg.BATCH_SIZE * _tg.GRAD_ACCUM})")
print(f"  Output     : {OUTPUT_DIR}")
print()

grpo_train(
    model_name   = MODEL_NAME,
    output_dir   = OUTPUT_DIR,
    n_episodes   = EPISODES,
    lora_rank    = LORA_RANK,
    seed         = SEED,
    run_eval     = False,   # separate eval cell below; skip here to save time
)

In [ ]:
# ── PLOT TRAINING CURVES ──────────────────────────────────────────────────────
import json
from pathlib import Path
from training.plot_rewards import plot_training_curves

curves_path = Path(OUTPUT_DIR) / "reward_curves.json"
if not curves_path.exists():
    print(f"[SKIP] {curves_path} not found — run the training cell first.")
else:
    curves = json.loads(curves_path.read_text())
    plots_dir = Path(OUTPUT_DIR) / "plots"
    plot_training_curves(curves, save_dir=str(plots_dir), show=True)
    print(f"Plots saved to {plots_dir}")

    # Print quick stats
    for role, vals in curves.items():
        if vals:
            n = len(vals)
            first_q = vals[:max(1, n//4)]
            last_q  = vals[max(0, 3*n//4):]
            trend = "↑" if sum(last_q)/len(last_q) > sum(first_q)/len(first_q) + 0.05 else "→"
            print(f"  {role:12s}: {sum(first_q)/len(first_q):.3f} → {sum(last_q)/len(last_q):.3f} {trend}")

In [ ]:
# ── INSPECT SAMPLE GENERATIONS ────────────────────────────────────────────────
# Reads generation_samples.jsonl written by RewardLogger every 50 steps.
# Check this for reward hacking — rising composite alone is not enough.
import json
from pathlib import Path

samples_path = Path(OUTPUT_DIR) / "generation_samples.jsonl"
if not samples_path.exists():
    print(f"[SKIP] No generation_samples.jsonl yet.")
else:
    lines = samples_path.read_text().strip().splitlines()
    print(f"Total logged samples: {len(lines)}\n")
    # Show last 4 samples
    for line in lines[-4:]:
        s = json.loads(line)
        print(f"  [{s['ts'][:16]}] role={s['role']:10s} reward={s['reward']:+.4f}")
        print(f"  completion preview: {s['completion'][:200]!r}")
        print()

In [ ]:
# ── BEFORE / AFTER EVAL ───────────────────────────────────────────────────────
# Compares heuristic baseline vs trained checkpoint on held-out episodes.
# Skip this cell for a quick demo — it re-runs inference on the trained model.
import json
from pathlib import Path
from training.eval import evaluate_model, print_comparison
from training.plot_rewards import plot_eval_comparison

EVAL_TASKS = ["delhi_monsoon_recovery_easy", "bengaluru_irrops_hard"]

print("Evaluating heuristic baseline...")
base = evaluate_model(
    model_name   = "heuristic-baseline",
    n_episodes   = EVAL_EPISODES,
    task_ids     = EVAL_TASKS,
    seed         = 99,
    use_generator= False,
    label        = "Heuristic Baseline",
)

ckpt = OUTPUT_DIR
print(f"\nEvaluating trained checkpoint: {ckpt}...")
trained = evaluate_model(
    model_name   = ckpt,
    n_episodes   = EVAL_EPISODES,
    task_ids     = EVAL_TASKS,
    seed         = 99,
    use_generator= False,
    label        = "GRPO Trained",
)

print_comparison(base, trained)

# Save + plot
out = {
    "base":    {k: v for k, v in base.items()    if k != "records"},
    "trained": {k: v for k, v in trained.items() if k != "records"},
}
eval_path = Path(OUTPUT_DIR) / "eval_results.json"
eval_path.write_text(json.dumps(out, indent=2))
print(f"\nSaved eval results → {eval_path}")

plots_dir = Path(OUTPUT_DIR) / "plots"
plot_eval_comparison(out, save_dir=str(plots_dir), show=True)

In [ ]:
# ── CHECKPOINT SUMMARY ────────────────────────────────────────────────────────
import os
from pathlib import Path

out = Path(OUTPUT_DIR)
files = list(out.rglob("*"))
print(f"Output directory: {OUTPUT_DIR}")
print(f"Total files     : {len(files)}")
key_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "reward_curves.json",
    "baseline_metrics.json",
    "generation_samples.jsonl",
    "eval_results.json",
]
print("\nKey files:")
for name in key_files:
    path = out / name
    status = f"{path.stat().st_size / 1024:.1f} KB" if path.exists() else "MISSING"
    print(f"  {name:<35} {status}")

print("\nAdapter config:")
adapter_cfg = out / "adapter_config.json"
if adapter_cfg.exists():
    import json
    cfg = json.loads(adapter_cfg.read_text())
    print(f"  base model   : {cfg.get('base_model_name_or_path')}")
    print(f"  lora_r       : {cfg.get('r')}")
    print(f"  target_modules: {cfg.get('target_modules')}")

## Troubleshooting

| Error | Fix |
|-------|-----|
| `No GPU found` | Runtime → Change runtime type → GPU |
| `GRPOTrainer not found` | Re-run the install cell (TRL may not have loaded yet) |
| `CUDA out of memory` | Set `N_GENERATIONS=1` in config; reduce `EPISODES` to 25 |
| `ModuleNotFoundError: training` | Re-run the clone cell (sys.path not set) |
| `branch not found` | Push your branch: `git push origin yashh` before cloning |
| `adapter_config.json MISSING` | Training didn't complete — check cell 8 output for errors |
| Rising reward but bad outputs | Check `generation_samples.jsonl` in cell 10 for reward hacking |
| Drive quota full | Set `USE_DRIVE=False` to use /content (lost on disconnect) |

**To load the checkpoint later:**
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "path/to/adapter",
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

**Do NOT** do `model.merge_and_unload()` on a 4-bit model — weights corrupt. Use `save_pretrained_merged` for a merged 16-bit checkpoint.